In [ ]:
# --- input layout shim (added automatically) --------------------------------
# Kaggle mounts attached data either at ../input/<slug>/ or, on API-pushed
# kernels, at /kaggle/input/datasets/<owner>/<slug>/ and
# /kaggle/input/competitions/<slug>/. learntools reads ../input/... itself at
# import time, so the path has to be made real rather than rewritten.
# /kaggle/input is read-only; /kaggle/working is not.
import glob as _glob
import os as _os

# Collect every mounted source, under either layout. The container
# directories themselves are skipped.
_mounted = [p for p in _glob.glob("/kaggle/input/*")
            if _os.path.isdir(p) and _os.path.basename(p) not in ("datasets", "competitions")]
_mounted += _glob.glob("/kaggle/input/datasets/*/*")
_mounted += _glob.glob("/kaggle/input/competitions/*")

# Always relocate, even when ../input/<slug> already resolves: /kaggle/input is
# read-only under BOTH layouts, and some exercises symlink a competition file
# to a bare ../input/train.csv before reading it. That write needs ../input to
# be ours.
# Must be literally "input": ../input from _nb resolves to /kaggle/working/input.
_farm = "/kaggle/working/input"
_here = "/kaggle/working/_nb"
_os.makedirs(_farm, exist_ok=True)
_os.makedirs(_here, exist_ok=True)
for _src in _mounted:
    _dst = _os.path.join(_farm, _os.path.basename(_src))
    if not _os.path.exists(_dst):
        _os.symlink(_src, _dst)
    # Some courses read a bare ../input/<file>.csv, because a single attached
    # dataset used to be mounted with its files directly under ../input. Expose
    # each dataset's own entries at the farm root as well so both spellings work.
    for _child in _glob.glob(_os.path.join(_src, "*")):
        _cdst = _os.path.join(_farm, _os.path.basename(_child))
        if not _os.path.exists(_cdst):
            _os.symlink(_child, _cdst)
_os.chdir(_here)


def _safe_symlink(src, dst):
    """Stand in for os.symlink in the course setup cells.

    Several exercises do

        if not os.path.exists("../input/x.csv"):
            os.symlink("../input/<slug>/x.csv", "../input/x.csv")

    which breaks two ways once ../input is the farm. If the shim already
    exposed x.csv, os.symlink raises FileExistsError -- os.path.exists returns
    False for a dangling link, so the guard does not protect it. And if the
    dataset did not mount, the call happily creates a dangling link and the
    read fails later with a confusing FileNotFoundError.

    Link only when the source is real and the name is free, and never raise.
    """
    try:
        if _os.path.lexists(dst):
            return
        if not _os.path.exists(src):
            return
        _os.symlink(src, dst)
    except OSError:
        pass


print("input shim active:", sorted(_os.listdir(_farm)))
# --- end shim ---------------------------------------------------------------


**This notebook is an exercise in the [Geospatial Analysis](https://www.kaggle.com/learn/geospatial-analysis) course.  You can reference the tutorial at [this link](https://www.kaggle.com/alexisbcook/your-first-map).**

---


# Introduction

[Kiva.org](https://www.kiva.org/) is an online crowdfunding platform extending financial services to poor people around the world. Kiva lenders have provided over $1 billion dollars in loans to over 2 million people.

<center>
<img src="https://storage.googleapis.com/kaggle-media/learn/images/2G8C53X.png" width="500"><br/>
</center>

Kiva reaches some of the most remote places in the world through their global network of "Field Partners". These partners are local organizations working in communities to vet borrowers, provide services, and administer loans.

In this exercise, you'll investigate Kiva loans in the Philippines.  Can you identify regions that might be outside of Kiva's current network, in order to identify opportunities for recruiting new Field Partners?

To get started, run the code cell below to set up our feedback system.

In [ ]:
import geopandas as gpd

from learntools.core import binder
binder.bind(globals())
from learntools.geospatial.ex1 import *

### 1) Get the data.

Use the next cell to load the shapefile located at `loans_filepath` to create a GeoDataFrame `world_loans`.  

In [ ]:
loans_filepath = "../input/geospatial-learn-course-data/kiva_loans/kiva_loans/kiva_loans.shp"

# Your code here: Load the data
world_loans = gpd.read_file(loans_filepath)

# Check your answer
q_1.check()

# Uncomment to view the first five rows of the data
#world_loans.head()


In [ ]:
# Lines below will give you a hint or solution code
#q_1.hint()
#q_1.solution()

### 2) Plot the data.

Run the next code cell without changes to load a GeoDataFrame `world` containing country boundaries.

In [ ]:
# This dataset is provided in GeoPandas
world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)
world.head()

Use the `world` and `world_loans` GeoDataFrames to visualize Kiva loan locations across the world.

In [ ]:
# Your code here
ax = world.plot(figsize=(20,20), color='whitesmoke', linestyle=':', edgecolor='black')
world_loans.plot(ax=ax, markersize=2)

# Uncomment to see a hint
#q_2.hint()


In [ ]:
# Get credit for your work after you have created a map
q_2.check()

# Uncomment to see our solution (your code may look different!)
#q_2.solution()

### 3) Select loans based in the Philippines.

Next, you'll focus on loans that are based in the Philippines.  Use the next code cell to create a GeoDataFrame `PHL_loans` which contains all rows from `world_loans` with loans that are based in the Philippines.

In [ ]:
# Your code here
PHL_loans = world_loans.loc[world_loans.country=="Philippines"].copy()

# Check your answer
q_3.check()


In [ ]:
# Lines below will give you a hint or solution code
#q_3.hint()
#q_3.solution()

### 4) Understand loans in the Philippines.

Run the next code cell without changes to load a GeoDataFrame `PHL` containing boundaries for all islands in the Philippines.

In [ ]:
# Load a KML file containing island boundaries
gpd.io.file.fiona.drvsupport.supported_drivers['KML'] = 'rw'
PHL = gpd.read_file("../input/geospatial-learn-course-data/Philippines_AL258.kml", driver='KML')
PHL.head()

Use the `PHL` and `PHL_loans` GeoDataFrames to visualize loans in the Philippines.

In [ ]:
# Your code here
ax = PHL.plot(figsize=(12,12), color='whitesmoke', linestyle=':', edgecolor='lightgray')
PHL_loans.plot(ax=ax, markersize=2)

# Uncomment to see a hint
#q_4.a.hint()


In [ ]:
# Get credit for your work after you have created a map
q_4.a.check()

# Uncomment to see our solution (your code may look different!)
#q_4.a.solution()

Can you identify any islands where it might be useful to recruit new Field Partners?  Do any islands currently look outside of Kiva's reach?

You might find [this map](https://bit.ly/2U2G7x7) useful to answer the question.

In [ ]:
# View the solution (Run this code cell to receive credit!)
q_4.b.solution()

# Keep going

Continue to learn about **[coordinate reference systems](https://www.kaggle.com/alexisbcook/coordinate-reference-systems)**.

---




*Have questions or comments? Visit the [course discussion forum](https://www.kaggle.com/learn/geospatial-analysis/discussion) to chat with other learners.*